# KELT-20b TESS Exploration With JAX / NumPyro

This notebook keeps the `lightkurve` download and detrending steps on CPU, then hands the transit fit to `jaxoplanet` + `numpyro`.

Notes:
- The preprocessing cells are still ordinary NumPy / `lightkurve` code.
- The modeling cells are written in JAX so they can run on CPU, CUDA, or Apple Metal.
- On Apple Metal, the model inputs are cast to `float32` because `jax-metal` does not currently support `float64`.
- The transit parameterization uses `TransitOrbit(duration=..., impact_param=..., radius_ratio=...)`, so this notebook fits duration directly instead of the `rho_star` parameterization used in the PyMC notebook.
        


In [ ]:
import lightkurve as lk
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

TARGET = "KELT-20"
MISSION = "TESS"
AUTHOR = "SPOC"
EXPTIME_S = 120

PERIOD_D = 3.4741039
T0_BTJD = 1698.210775
TRANSIT_DURATION_D = 3.54 / 24.0
PLOT_WINDOW_D = 0.15
MODEL_WINDOW_D = PLOT_WINDOW_D
RADIUS_RATIO_GUESS = 0.116
B_GUESS = 0.60

search = lk.search_lightcurve(TARGET, mission=MISSION, author=AUTHOR, exptime=EXPTIME_S)
display(search)

lc_collection = search.download_all()
print(f"Downloaded {len(lc_collection)} light curves")
        


In [ ]:
stitched_lc = lc_collection.stitch().remove_nans().normalize()
transit_mask = stitched_lc.create_transit_mask(
    period=PERIOD_D,
    transit_time=T0_BTJD,
    duration=TRANSIT_DURATION_D,
)
flat_lc = stitched_lc.flatten(window_length=901, mask=transit_mask)
folded_lc = flat_lc.fold(period=PERIOD_D, epoch_time=T0_BTJD)
binned_folded_lc = folded_lc.bin(time_bin_size=5.0 / (24.0 * 60.0))

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False)

stitched_lc.scatter(ax=axes[0], color="0.35", alpha=0.35, s=4)
axes[0].set_title("KELT-20 stitched TESS light curve")
axes[0].set_ylabel("Normalized flux")

folded_lc.scatter(ax=axes[1], color="0.6", alpha=0.18, s=6, label="Unbinned")
binned_folded_lc.scatter(ax=axes[1], color="tab:blue", s=18, label="5 min bins")
axes[1].set_xlim(-PLOT_WINDOW_D, PLOT_WINDOW_D)
axes[1].set_title("Phase-folded transit")
axes[1].set_xlabel("Time from mid-transit [days]")
axes[1].set_ylabel("Normalized flux")
axes[1].legend()

plt.tight_layout()
        


In [ ]:
sector_lc = lc_collection[0].remove_nans().normalize()
sector_transit_mask = sector_lc.create_transit_mask(
    period=PERIOD_D,
    transit_time=T0_BTJD,
    duration=TRANSIT_DURATION_D,
)
sector_flat_lc = sector_lc.flatten(window_length=901, mask=sector_transit_mask)
sector_folded_lc = sector_flat_lc.fold(period=PERIOD_D, epoch_time=T0_BTJD)

fig, ax = plt.subplots(figsize=(10, 5))
sector_folded_lc.plot_river(ax=ax)
ax.set_aspect("auto")
ax.set_title("KELT-20b river plot for the first downloaded TESS sector")
plt.tight_layout()
        


In [ ]:
time_btjd = flat_lc.time.value
phase_days = ((time_btjd - T0_BTJD + 0.5 * PERIOD_D) % PERIOD_D) - 0.5 * PERIOD_D
epoch_number = np.round((time_btjd - T0_BTJD) / PERIOD_D).astype(int)

fig, ax = plt.subplots(figsize=(11, 6))
unique_epochs = np.unique(epoch_number)
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_epochs)))

for color, current_epoch in zip(colors, unique_epochs):
    mask = (epoch_number == current_epoch) & (np.abs(phase_days) < PLOT_WINDOW_D)
    if not np.any(mask):
        continue
    ax.errorbar(
        phase_days[mask],
        flat_lc.flux.value[mask],
        yerr=flat_lc.flux_err.value[mask],
        fmt="o",
        ms=4,
        color=color,
        ecolor="0.75",
        elinewidth=0.8,
        capsize=0,
        alpha=0.85,
        label=f"Orbit {current_epoch}",
    )

ax.set_title("Epoch-to-epoch transit overlay")
ax.set_xlabel("Time from mid-transit [days]")
ax.set_ylabel("Normalized flux")
ax.grid(alpha=0.25)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, ncol=1)
plt.tight_layout()
        


## Build A CPU-Preprocessed Joint Dataset

The next cell mirrors the original notebook's sector-level detrending, but it only prepares NumPy arrays. The JAX model starts after this step.
        


In [ ]:
def build_joint_tess_dataset(lc_collection, period_d, t0_btjd, transit_duration_d, model_window_d):
    sector_labels = []
    x_parts = []
    y_parts = []
    yerr_parts = []
    sector_idx_parts = []

    for sector_number, raw_lc in enumerate(lc_collection, start=1):
        sector_label = raw_lc.meta.get("MISSION", f"sector_{sector_number}")
        sector_lc = raw_lc.remove_nans().normalize().remove_outliers(sigma=5.0)
        transit_mask = sector_lc.create_transit_mask(
            period=period_d,
            transit_time=t0_btjd,
            duration=transit_duration_d,
        )
        sector_flat = sector_lc.flatten(window_length=901, mask=transit_mask)

        sector_phase = ((sector_flat.time.value - t0_btjd + 0.5 * period_d) % period_d) - 0.5 * period_d
        window_mask = np.abs(sector_phase) < model_window_d
        if not np.any(window_mask):
            continue

        flux = np.asarray(sector_flat.flux.value, dtype=np.float64)
        flux_err = np.asarray(sector_flat.flux_err.value, dtype=np.float64)

        finite_flux = flux[np.isfinite(flux)]
        flux_fill = float(np.median(finite_flux)) if finite_flux.size else 1.0
        flux = np.where(np.isfinite(flux), flux, flux_fill)

        finite_err = flux_err[np.isfinite(flux_err) & (flux_err > 0)]
        err_fill = float(np.median(finite_err)) if finite_err.size else 5.0e-4
        flux_err = np.where(np.isfinite(flux_err) & (flux_err > 0), flux_err, err_fill)

        x_parts.append(np.ascontiguousarray(sector_flat.time.value[window_mask], dtype=np.float64))
        y_parts.append(np.ascontiguousarray(flux[window_mask] - 1.0, dtype=np.float64))
        yerr_parts.append(np.ascontiguousarray(flux_err[window_mask], dtype=np.float64))
        sector_idx_parts.append(np.full(np.count_nonzero(window_mask), len(sector_labels), dtype=np.int32))
        sector_labels.append(sector_label)

    if not x_parts:
        raise RuntimeError("No TESS cadences survived the preprocessing window.")

    x = np.concatenate(x_parts)
    y = np.concatenate(y_parts)
    yerr = np.concatenate(yerr_parts)
    sector_idx = np.concatenate(sector_idx_parts)

    sort_idx = np.argsort(x)
    x = x[sort_idx]
    y = y[sort_idx]
    yerr = yerr[sort_idx]
    sector_idx = sector_idx[sort_idx]

    n_sectors = len(sector_labels)
    sector_counts = np.bincount(sector_idx, minlength=n_sectors)

    return {
        "time": x,
        "flux": y,
        "flux_err": yerr,
        "sector_idx": sector_idx,
        "sector_labels": sector_labels,
        "sector_counts": sector_counts,
        "n_sectors": n_sectors,
    }


dataset = build_joint_tess_dataset(
    lc_collection=lc_collection,
    period_d=PERIOD_D,
    t0_btjd=T0_BTJD,
    transit_duration_d=TRANSIT_DURATION_D,
    model_window_d=MODEL_WINDOW_D,
)

print(f"Prepared {dataset['time'].size} cadences across {dataset['n_sectors']} sectors")
for label, count in zip(dataset["sector_labels"], dataset["sector_counts"]):
    print(f"  {label}: {int(count)} cadences in the transit window")
        


## JAX / NumPyro Transit Model

This model uses `jaxoplanet.orbits.TransitOrbit` and a direct NumPyro NUTS sampler. To stay Metal-compatible, the notebook drops to `float32` when the active JAX device reports a Metal backend.
        


In [ ]:
import arviz as az
import jax

jax.config.update("jax_enable_x64", True)

JAX_PLATFORM = jax.devices()[0].platform.lower()
IS_METAL = "metal" in JAX_PLATFORM

import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from jaxoplanet.light_curves import limb_dark_light_curve
from jaxoplanet.orbits import TransitOrbit
from numpyro.infer import MCMC, NUTS, init_to_value

MODEL_DTYPE = jnp.float32 if IS_METAL else jnp.float64
NP_DTYPE = np.float32 if IS_METAL else np.float64
LC_ORDER = 20

print("JAX devices:", jax.devices())
print("Default backend:", jax.default_backend())
if IS_METAL:
    print("Metal backend detected; using float32 model arrays for jax-metal compatibility.")
else:
    print("Non-Metal backend detected; using float64 model arrays.")
        


In [ ]:
def kipping_q_to_u(q1, q2):
    sqrt_q1 = jnp.sqrt(jnp.clip(q1, 0.0, 1.0))
    u1 = 2.0 * sqrt_q1 * q2
    u2 = sqrt_q1 * (1.0 - 2.0 * q2)
    return jnp.stack([u1, u2])


def transit_flux_model(t, period, duration, t0, b, r, u, order=LC_ORDER):
    orbit = TransitOrbit(
        period=period,
        duration=duration,
        time_transit=t0,
        impact_param=b,
        radius_ratio=r,
    )
    return limb_dark_light_curve(orbit, u, order=order)(t)


def joint_tess_model(t, yerr, sector_idx, n_sectors, y=None):
    period_loc = jnp.asarray(PERIOD_D, dtype=MODEL_DTYPE)
    t0_loc = jnp.asarray(T0_BTJD, dtype=MODEL_DTYPE)
    duration_loc = jnp.asarray(TRANSIT_DURATION_D, dtype=MODEL_DTYPE)
    jitter_loc = jnp.asarray(np.median(dataset["flux_err"]), dtype=MODEL_DTYPE)

    log_period = numpyro.sample(
        "log_period",
        dist.Normal(jnp.log(period_loc), jnp.asarray(2.0e-4 / PERIOD_D, dtype=MODEL_DTYPE)),
    )
    period = numpyro.deterministic("period", jnp.exp(log_period))

    t0 = numpyro.sample("t0", dist.Normal(t0_loc, jnp.asarray(3.0e-3, dtype=MODEL_DTYPE)))

    log_duration = numpyro.sample(
        "log_duration",
        dist.Normal(jnp.log(duration_loc), jnp.asarray(0.15, dtype=MODEL_DTYPE)),
    )
    duration = numpyro.deterministic("duration", jnp.exp(log_duration))

    r = numpyro.sample(
        "r",
        dist.Uniform(jnp.asarray(0.01, dtype=MODEL_DTYPE), jnp.asarray(0.25, dtype=MODEL_DTYPE)),
    )
    b_scaled = numpyro.sample(
        "b_scaled",
        dist.Uniform(jnp.asarray(0.0, dtype=MODEL_DTYPE), jnp.asarray(1.0, dtype=MODEL_DTYPE)),
    )
    b = numpyro.deterministic("b", b_scaled * (1.0 + r))

    q1 = numpyro.sample(
        "q1",
        dist.Uniform(jnp.asarray(0.0, dtype=MODEL_DTYPE), jnp.asarray(1.0, dtype=MODEL_DTYPE)),
    )
    q2 = numpyro.sample(
        "q2",
        dist.Uniform(jnp.asarray(0.0, dtype=MODEL_DTYPE), jnp.asarray(1.0, dtype=MODEL_DTYPE)),
    )
    u = kipping_q_to_u(q1, q2)
    numpyro.deterministic("u1", u[0])
    numpyro.deterministic("u2", u[1])

    mean_flux = numpyro.sample(
        "mean_flux",
        dist.Normal(
            jnp.zeros((n_sectors,), dtype=MODEL_DTYPE),
            jnp.full((n_sectors,), 5.0e-4, dtype=MODEL_DTYPE),
        ),
    )
    log_jitter = numpyro.sample(
        "log_jitter",
        dist.Normal(
            jnp.full((n_sectors,), jnp.log(jitter_loc), dtype=MODEL_DTYPE),
            jnp.full((n_sectors,), 2.0, dtype=MODEL_DTYPE),
        ),
    )
    jitter = jnp.exp(log_jitter)

    model_flux = transit_flux_model(t=t, period=period, duration=duration, t0=t0, b=b, r=r, u=u)
    mu = model_flux + mean_flux[sector_idx]
    obs_sigma = jnp.sqrt(yerr**2 + jitter[sector_idx]**2)

    numpyro.deterministic("transit_depth", r**2)
    numpyro.deterministic("transit_depth_percent", 100.0 * r**2)
    numpyro.sample("obs", dist.Normal(mu, obs_sigma), obs=y)
        


In [ ]:
jax_data = {
    "t": jnp.asarray(dataset["time"], dtype=MODEL_DTYPE),
    "y": jnp.asarray(dataset["flux"], dtype=MODEL_DTYPE),
    "yerr": jnp.asarray(dataset["flux_err"], dtype=MODEL_DTYPE),
    "sector_idx": jnp.asarray(dataset["sector_idx"], dtype=jnp.int32),
}

init_values = {
    "log_period": np.asarray(np.log(PERIOD_D), dtype=NP_DTYPE),
    "t0": np.asarray(T0_BTJD, dtype=NP_DTYPE),
    "log_duration": np.asarray(np.log(TRANSIT_DURATION_D), dtype=NP_DTYPE),
    "r": np.asarray(RADIUS_RATIO_GUESS, dtype=NP_DTYPE),
    "b_scaled": np.asarray(B_GUESS / (1.0 + RADIUS_RATIO_GUESS), dtype=NP_DTYPE),
    "q1": np.asarray(0.25, dtype=NP_DTYPE),
    "q2": np.asarray(0.25, dtype=NP_DTYPE),
    "mean_flux": np.zeros(dataset["n_sectors"], dtype=NP_DTYPE),
    "log_jitter": np.full(dataset["n_sectors"], np.log(np.median(dataset["flux_err"])), dtype=NP_DTYPE),
}

sampler = MCMC(
    NUTS(
        joint_tess_model,
        target_accept_prob=0.9,
        init_strategy=init_to_value(values=init_values),
    ),
    num_warmup=1000,
    num_samples=1000,
    num_chains=2,
    chain_method="sequential",
    progress_bar=True,
)

sampler.run(
    jax.random.PRNGKey(1),
    t=jax_data["t"],
    yerr=jax_data["yerr"],
    sector_idx=jax_data["sector_idx"],
    n_sectors=dataset["n_sectors"],
    y=jax_data["y"],
)

idata = az.from_numpyro(sampler)
        


In [ ]:
summary_vars = [
    "period",
    "t0",
    "duration",
    "r",
    "b",
    "u1",
    "u2",
    "transit_depth_percent",
]

az.summary(idata, var_names=summary_vars, round_to=6)
        


In [ ]:
az.plot_trace(idata, var_names=summary_vars)
plt.tight_layout()
        


In [ ]:
posterior = idata.posterior

period_med = posterior["period"].median(dim=("chain", "draw")).item()
t0_med = posterior["t0"].median(dim=("chain", "draw")).item()
duration_med = posterior["duration"].median(dim=("chain", "draw")).item()
r_med = posterior["r"].median(dim=("chain", "draw")).item()
b_med = posterior["b"].median(dim=("chain", "draw")).item()
u_med = np.array(
    [
        posterior["u1"].median(dim=("chain", "draw")).item(),
        posterior["u2"].median(dim=("chain", "draw")).item(),
    ],
    dtype=NP_DTYPE,
)
mean_flux_med = posterior["mean_flux"].median(dim=("chain", "draw")).values.astype(NP_DTYPE)
jitter_med = np.exp(posterior["log_jitter"].median(dim=("chain", "draw")).values).astype(NP_DTYPE)

transit_model_med = np.asarray(
    transit_flux_model(
        t=jax_data["t"],
        period=jnp.asarray(period_med, dtype=MODEL_DTYPE),
        duration=jnp.asarray(duration_med, dtype=MODEL_DTYPE),
        t0=jnp.asarray(t0_med, dtype=MODEL_DTYPE),
        b=jnp.asarray(b_med, dtype=MODEL_DTYPE),
        r=jnp.asarray(r_med, dtype=MODEL_DTYPE),
        u=jnp.asarray(u_med, dtype=MODEL_DTYPE),
    )
)
mean_model_med = transit_model_med + mean_flux_med[dataset["sector_idx"]]
detrended_flux = dataset["flux"] - mean_flux_med[dataset["sector_idx"]]
residual_flux = dataset["flux"] - mean_model_med

fig, axes = plt.subplots(3, 1, figsize=(11, 11), sharex=True)

axes[0].plot(dataset["time"], dataset["flux"], ".", color="0.4", alpha=0.22, ms=4, label="Data")
axes[0].plot(dataset["time"], mean_model_med, color="tab:red", lw=2, label="Transit + sector offset")
axes[0].set_title("Joint multi-sector raw light curve fit")
axes[0].set_ylabel("Flux - 1")
axes[0].legend()

axes[1].plot(dataset["time"], detrended_flux, ".", color="0.35", alpha=0.22, ms=4, label="Offset-corrected data")
axes[1].plot(dataset["time"], transit_model_med, color="tab:blue", lw=2, label="Transit model")
axes[1].set_title("Offset-corrected light curve")
axes[1].set_ylabel("Detrended flux")
axes[1].legend()

axes[2].plot(dataset["time"], residual_flux, ".", color="0.4", alpha=0.25, ms=4)
axes[2].axhline(0.0, color="tab:red", lw=1.5)
axes[2].set_title("Residuals")
axes[2].set_xlabel("Time [BTJD]")
axes[2].set_ylabel("Residual flux")

plt.tight_layout()

print("Median sector offsets:")
for label, offset, jitter in zip(dataset["sector_labels"], mean_flux_med, jitter_med):
    print(f"  {label}: offset={offset:+.6e}, jitter={jitter:.6e}")
        


In [ ]:
def median_pm(samples):
    samples = np.asarray(samples).reshape(-1)
    q16, q50, q84 = np.percentile(samples, [16, 50, 84])
    return q50, q84 - q50, q50 - q16


phase_days = ((dataset["time"] - t0_med + 0.5 * period_med) % period_med) - 0.5 * period_med
sort_idx = np.argsort(phase_days)

bin_edges = np.linspace(-PLOT_WINDOW_D, PLOT_WINDOW_D, 81)
bin_index = np.digitize(phase_days, bin_edges)
binned_phase = []
binned_flux = []
for idx in range(1, len(bin_edges)):
    mask = bin_index == idx
    if np.count_nonzero(mask) < 3:
        continue
    binned_phase.append(np.mean(phase_days[mask]))
    binned_flux.append(np.mean(detrended_flux[mask]))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(phase_days, detrended_flux, ".", color="0.5", alpha=0.18, ms=4, label="All detrended cadences")
ax.plot(np.asarray(binned_phase), np.asarray(binned_flux), "o", color="tab:blue", ms=5, label="Binned data")
ax.plot(phase_days[sort_idx], transit_model_med[sort_idx], color="tab:red", lw=2, label="Transit model")
ax.set_xlim(-PLOT_WINDOW_D, PLOT_WINDOW_D)
ax.set_title("Phase-folded joint multi-sector transit")
ax.set_xlabel("Time from mid-transit [days]")
ax.set_ylabel("Detrended flux")
ax.legend()
plt.tight_layout()

r_stats = median_pm(posterior["r"].values)
b_stats = median_pm(posterior["b"].values)
duration_stats = median_pm(posterior["duration"].values)
depth_stats = median_pm(posterior["transit_depth_percent"].values)

print(f"Rp/R* = {r_stats[0]:.5f} +{r_stats[1]:.5f} -{r_stats[2]:.5f}")
print(f"b = {b_stats[0]:.4f} +{b_stats[1]:.4f} -{b_stats[2]:.4f}")
print(f"Duration [d] = {duration_stats[0]:.5f} +{duration_stats[1]:.5f} -{duration_stats[2]:.5f}")
print(f"Transit depth (%) = {depth_stats[0]:.4f} +{depth_stats[1]:.4f} -{depth_stats[2]:.4f}")
print("Sectors fit:")
for label, count in zip(dataset["sector_labels"], dataset["sector_counts"]):
    print(f"  {label}: {int(count)} cadences")
        
